In [469]:
import sympy as sp
from sympy import *
from IPython.display import display, Math

def print_math(str):
    display(Math(str))

Q1: Derive jacobian matrix.

In [470]:
# Symbols
rho, u, E ,gamma= sp.symbols('rho u E gamma')
U1, U2, U3 = sp.symbols('U1 U2 U3')
U_sym = sp.Matrix([U1, U2, U3])

# Pressure
# p = (gamma - 1) * (U3 - sp.Rational(1,2) * U2**2 / U1)
p=sp.symbols("p")
p_subs = {
    p: (gamma - 1) * (U3 - sp.Rational(1,2) * U2**2 / U1)
}

# Flux components
F1 = U2
F2 = U1 * (U2/U1)**2 + p
F3 = (U2/U1) * (U3 + p)

# Flux vector
F = sp.Matrix([F1, F2, F3])
F

Matrix([
[            U2],
[  p + U2**2/U1],
[U2*(U3 + p)/U1]])

In [471]:
A = F.subs(p_subs).jacobian(U_sym)
U_subs = {
    U1: rho,
    U2: rho*u,
    U3: rho*E
}

cv, T, a, R = sp.symbols('cv T a R')
thermo_subs = {
    E:cv*T + u**2/2,
    cv:R/(gamma - 1),
    T: a**2 / (gamma * R)
}

A = sp.simplify(A.subs(U_subs).subs(thermo_subs))

A

Matrix([
[                                                                  0,                                                              1,         0],
[                                                 u**2*(gamma - 3)/2,                                                  u*(3 - gamma), gamma - 1],
[u*(-2*a**2 + gamma**2*u**2 - 3*gamma*u**2 + 2*u**2)/(2*(gamma - 1)), (a**2 - gamma**2*u**2 + 5*gamma*u**2/2 - 3*u**2/2)/(gamma - 1),   gamma*u]])

In [472]:
U =  U_sym.subs(U_subs).subs(thermo_subs)
print_math("U =")
U

<IPython.core.display.Math object>

Matrix([
[                                    rho],
[                                  rho*u],
[rho*(a**2/(gamma*(gamma - 1)) + u**2/2)]])

Q2: Derive right eigenvectors. QA = [r1 r2 r3]

In [473]:
eigenvalues = [val for val, _, _ in A.eigenvects()]
eigenvectors = [v for _, _, vs in A.eigenvects() for v in vs]

# rescaling and reording eigenvectors to be consistent with notes
r1= eigenvectors[0]*(1/eigenvectors[0][0])
r2= eigenvectors[2]*(1/eigenvectors[2][0]) 
r3= -eigenvectors[1]*(1/eigenvectors[1][0])

print("Expanded, unscaled by rho/(2a)")
print_math("Q_A=")
Matrix.hstack(r1, r2, r3)


Expanded, unscaled by rho/(2a)


<IPython.core.display.Math object>

Matrix([
[     1,                                                                1,                                                                -1],
[     u,                (2*a*gamma - 2*a + 2*gamma*u - 2*u)/(2*gamma - 2),               -(-2*a*gamma + 2*a + 2*gamma*u - 2*u)/(2*gamma - 2)],
[u**2/2, (2*a**2 + 2*a*gamma*u - 2*a*u + gamma*u**2 - u**2)/(2*gamma - 2), -(2*a**2 - 2*a*gamma*u + 2*a*u + gamma*u**2 - u**2)/(2*gamma - 2)]])

In [474]:
# collect and simplify entries of r1, r2, r3 
def clean_r(v):
    return Matrix([collect((simplify(entry)), [u**2/2,  a**2,u*a]) for entry in v])
# scale r2 and r3 by rho/(2a) to be consistent with notes
r1=clean_r(r1)
r2=clean_r(r2)*rho/(2*a)
r3=clean_r(r3)*rho/(2*a)
Qa = Matrix.hstack(r1, r2, r3)

print_math("Q_A=")
Qa

<IPython.core.display.Math object>

Matrix([
[     1,                                                           rho/(2*a),                                                           -rho/(2*a)],
[     u,                                                   rho*(a + u)/(2*a),                                                    rho*(a - u)/(2*a)],
[u**2/2, rho*(a**2 + a*u*(gamma - 1) + u**2*(gamma - 1)/2)/(2*a*(gamma - 1)), rho*(-a**2 + a*u*(gamma - 1) + u**2*(1 - gamma)/2)/(2*a*(gamma - 1))]])

Q3: Derive flux-vector splitting

In [475]:
l1, l2, l3 = sp.symbols('l1 l2 l3')

Lambda = sp.diag(l1, l2, l3)
lambda_subs = {
    l1: eigenvalues[0],
    l2: eigenvalues[1],
    l3: eigenvalues[2]
}

print_math("\\Lambda =")
Lambda.subs(lambda_subs)

<IPython.core.display.Math object>

Matrix([
[u,      0,     0],
[0, -a + u,     0],
[0,      0, a + u]])

In [476]:
Qa_inv = Qa.inv()
print_math("Q_A^{-1}=")
Qa_inv

<IPython.core.display.Math object>

Matrix([
[ (2*a**2 - gamma*u**2 + u**2)/(2*a**2),        (gamma*u - u)/a**2,    (1 - gamma)/a**2],
[(-2*a*u + gamma*u**2 - u**2)/(2*a*rho), (a - gamma*u + u)/(a*rho), (gamma - 1)/(a*rho)],
[(-2*a*u - gamma*u**2 + u**2)/(2*a*rho), (a + gamma*u - u)/(a*rho), (1 - gamma)/(a*rho)]])

In [477]:
F_pm =Qa*Lambda*Qa_inv*U
F_pm_factored = Matrix([factor(collect(row, [l1, l2, l3])) for row in F_pm/(rho/(2*gamma))])
F_pm_factored[2] = collect((F_pm_factored[2])*2*(gamma-1), [l1, l2, l3])/(2*(gamma-1))
print_math("F^{\\pm}=\\frac{\\rho}{2\\gamma}" + sp.latex(F_pm_factored)) 


<IPython.core.display.Math object>

In [478]:
F_pm_factored[2]
expr =(gamma - 1)*l1*u**2 \
       + sp.Rational(1, 2)*l2*(u + a)**2 \
       + sp.Rational(1, 2)*l3*(u - a)**2 \
       + (3 - gamma)/(2*(gamma - 1))*(l2 + l3)*a**2
(F_pm_factored[2]-expr).simplify()
F_pm_factored[2]

(l1*(2*gamma**2*u**2 - 4*gamma*u**2 + 2*u**2) + l2*(2*a**2 + 2*a*gamma*u - 2*a*u + gamma*u**2 - u**2) + l3*(2*a**2 - 2*a*gamma*u + 2*a*u + gamma*u**2 - u**2))/(2*gamma - 2)

In [479]:
Lambda_QaInv_U= simplify(Lambda*Qa_inv*U) 
r1*Lambda_QaInv_U[0] + r2*Lambda_QaInv_U[1] + r3*Lambda_QaInv_U[2]


Matrix([
[                                                                                                                              l1*rho*(gamma - 1)/gamma + l2*rho/(2*gamma) + l3*rho/(2*gamma)],
[                                                                                                            l1*rho*u*(gamma - 1)/gamma + l2*rho*(a + u)/(2*gamma) - l3*rho*(a - u)/(2*gamma)],
[l1*rho*u**2*(gamma - 1)/(2*gamma) + l2*rho*(a**2 + a*u*(gamma - 1) + u**2*(gamma - 1)/2)/(2*gamma*(gamma - 1)) - l3*rho*(-a**2 + a*u*(gamma - 1) + u**2*(1 - gamma)/2)/(2*gamma*(gamma - 1))]])

In [480]:
Lambda_QaInv_U= simplify(Lambda*Qa_inv*U) 
print_math("F^{\\pm} = \\frac{1}{\\gamma}Q_A" + sp.latex(Lambda_QaInv_U))

print_math("F^{\\pm} = " + sp.latex(Lambda_QaInv_U[0]) + sp.latex(r1)+ "+" + sp.latex(Lambda_QaInv_U[1]) + sp.latex(r2) + "+" + sp.latex(Lambda_QaInv_U[2]) + sp.latex(r3))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [481]:
Lambda_QaInv_U

Matrix([
[l1*rho*(gamma - 1)/gamma],
[              a*l2/gamma],
[             -a*l3/gamma]])

Q4: Leer's flux vector

f1 = rho*u; mass flux

In [516]:
M = sp.symbols('M')
a1, b1, c1 = sp.symbols('a1 b1 c1')
a2, b2, c2 = sp.symbols('a2 b2 c2')
m1_p = a1*M**2 + b1*M + c1
m1_m = a2*M**2 + b2*M + c2
# f1 = (-1)*a*rho = 
# F+ moving to right
f1 = F[0].subs(U_subs).subs({u: M*a})
f1_p =  f1.subs({M: m1_p})
f1_m =  f1.subs({M: m1_m})

# 1. M = 1: fully supersonic to right
eq1 = sp.Eq(f1_p.subs(M, 1), f1.subs(M,1))  # f1+ @M=1 = f => m+ = 1
# 2. M = -1: fully supersonic to the left
eq2 = sp.Eq(f1_m.subs(M, -1), f1.subs(M,-1))  # f1-= @M=-1 = 0 => m- =0
# 3. M = -1: smoothness
eq3 = sp.Eq(sp.diff(f1_p, M).subs(M, -1), 0) # d(f1+)/dM @ M=-1 = 0 
# 4. M = 1: smoothness
eq4 = sp.Eq(sp.diff(f1_m, M).subs(M, 1), 0)  # d(f1-)/dM @ M=1 = 0 
# 5, 6.  conservation
eq5 = sp.Eq(a1+a2, 0)
eq6 = sp.Eq(b1+b2, 1)
sol = sp.solve([eq1, eq2, eq3, eq4, eq5, eq6], [a1, b1, c1, a2, b2, c2])
# sol = sp.solve([eq1, eq2, eq3], [a1, b1, c1])
f1_p = f1_p.subs(sol).factor()
f1_m = f1_m.subs(sol).factor()
print_math("f_1^+ = " + sp.latex(f1_p))
print_math("f_1^- = " + sp.latex(f1_m))
m1_p = m1_p.subs(sol).factor()
m1_m = m1_m.subs(sol).factor()    
print_math("M^+ = " + sp.latex(m1_p))
print_math("M^- = " + sp.latex(m1_m))


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

f2 = rho*u**2 + p; momentum flux

In [483]:
p_expanded = p.subs(p_subs).subs(U_subs).subs(thermo_subs).simplify()

# f2 = rho*u**2 + p; momentum flux
a1, b1, c1, d1 = sp.symbols('a1 b1 c1 d1')
a2, b2, c2, d2 = sp.symbols('a2 b2 c2 d2')

f2 = F[1].subs({u: M*a})
m = f2-p

m2_p = m.subs({M**2: m1_p*(a1*M + b1)})
m2_m = m.subs({M**2: m1_m*(a2*M + b2)})
M_expr = collect(((m2_m + m2_p)/(a**2*rho)).expand().factor(),M)
print_math("\\frac{m^+ + m^-}{\\rho a^2}="+ sp.latex(M_expr) + "=M^2")
# 1. M**3
eq1 = sp.Eq((a1-a2)/4,0)
# 2. M**2
eq2 = sp.Eq((2*a1 + 2*a2 + b1 - b2)/4,1)
# 3. M
eq3 = sp.Eq((a1-a2+2*b1+2*b2)/4,0)
# 4. 0M
eq4 = sp.Eq((b1-b2)/4,0)

p_p = m1_p*(c1*M+d1)*p
p_m = m1_m*(c2*M+d2)*p
p_expr = sp.collect(((p_p+p_m)/p).factor(), M)
print_math("\\frac{p}{p}=\\frac{p^+ + p^-}{p} = " + sp.latex(p_expr) + "= 1")
# 5. M**3
eq5 = sp.Eq((c1-c2)/4, 0)
# 6. M**2
eq6 = sp.Eq((2*c1 + 2*c2 + d1 - d2)/4, 0)
# 7. M
eq7 = sp.Eq((c1 -c2 +2*d1 + 2*d2)/4, 0)
# 8. 0
eq8 = sp.Eq((d1-d2)/4, 1)

sol = sp.solve([eq1, eq2, eq3, eq4, eq5, eq6, eq7, eq8], [a1, b1, c1, d1, a2, b2, c2, d2])
print_math("sol="+sp.latex(sol))
m2_p = m2_p.subs(sol)
m2_m = m2_m.subs(sol)
p_p = p_p.subs(sol)
p_m = p_m.subs(sol)
f2_p = sp.simplify(((m2_p.subs(sol) + p_p.subs(sol))))
f2_m = sp.simplify(((m2_m.subs(sol) + p_m.subs(sol))))

f2_p_factored = sp.collect(((m2_p.subs(sol) + p_p.subs(sol))/f1_p).subs({M:u/a}).subs({p:p_expanded}).simplify(), u)
f2_m_factored = sp.collect(((m2_m.subs(sol) + p_m.subs(sol))/f1_m).subs({M:u/a}).subs({p:p_expanded}).simplify(), u)
print_math("f_2^+ = " + sp.latex(f2_p) +"=" + sp.latex(f1_p) +"\\cdot"+sp.latex(f2_p_factored))
print_math("f_2^- = " + sp.latex(f2_m)+"=" + sp.latex(f1_p) +"\\cdot"+sp.latex(f2_m_factored))
m


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

U2**2/U1

f3 = u*(E*rho + p); energy flux

In [484]:
D = sp.diag( (gamma-1)*rho, a, -a)
D_inv = D.inv()
B = sp.simplify(D_inv*Qa_inv)
B

Matrix([
[(2*a**2 - gamma*u**2 + u**2)/(2*a**2*rho*(gamma - 1)),                  u/(a**2*rho),          -1/(a**2*rho)],
[                  u*(-2*a + gamma*u - u)/(2*a**2*rho),  (a - gamma*u + u)/(a**2*rho), (gamma - 1)/(a**2*rho)],
[                   u*(2*a + gamma*u - u)/(2*a**2*rho), (-a - gamma*u + u)/(a**2*rho), (gamma - 1)/(a**2*rho)]])

In [ ]:
B_collected = (B*(rho*a**2)).applyfunc(lambda x: sp.collect(sp.apart(x, gamma), [u**2/2,u,]))
B_collected
lambdas = sp.Matrix([l1, l2, l3])
factor = gamma/(rho*a**2) 
print_math(
    sp.latex(lambdas) + "="
    + sp.latex(factor) + sp.latex(B_collected[:,0]) + "F_1"+ "+" 
    + sp.latex(factor) + sp.latex(B_collected[:,1]) + "F_2" + "+" 
    + sp.latex(factor) + sp.latex(B_collected[:,2]) + "F_3"
)

<IPython.core.display.Math object>

In [486]:
M = sp.symbols("M")
F_vl_p = (rho*a/4)*(M+1)**2 * sp.Matrix([
        1,
        ((gamma-1)*u+2*a)/gamma,
        ((gamma-1)*u+2*a)**2/(2*(gamma+1)*(gamma-1))
    ])

F_vl_m = (rho*a/4)*(M-1)**2 * sp.Matrix([
        1,
        ((gamma-1)*u-2*a)/gamma,
        ((gamma-1)*u-2*a)**2/(2*(gamma+1)*(gamma-1))
    ])

D = sp.diag( (gamma-1)*rho, a, -a)
D_inv = D.inv()
B = sp.simplify(D_inv*Qa_inv)
lambdas_vl_p = gamma*B*F_vl_p
lambdas_vl_m = gamma*B*F_vl_m
# lambdas_vl_p.applyfunc(lambda x: sp.collect(x.expand(), [u**2, a*u,a**2]))
lambdas_vl_m
F_vl_p

Matrix([
[                                                     a*rho*(M + 1)**2/4],
[                       a*rho*(M + 1)**2*(2*a + u*(gamma - 1))/(4*gamma)],
[a*rho*(M + 1)**2*(2*a + u*(gamma - 1))**2/(4*(gamma - 1)*(2*gamma + 2))]])

In [579]:
f1_p_sym = sp.symbols("f_1^+")
F_pp = sp.IndexedBase('F^p+')


B = sp.simplify(D_inv*Qa_inv)*gamma
factor = 1/(rho*a**2)
B_collected = (B/factor).applyfunc(lambda x: sp.collect(sp.apart(x, gamma), [u**2/2,u,]))
l_p_vl = sp.Matrix([0, 0, 0])
i=0

B_row = B_collected[i,:]
F_factored = F_vl_p/F_vl_p[i]
expr=(B_row[0]*F_factored[0] 
    + B_row[1]*F_factored[1] 
    + B_row[2]*F_factored[2]
    )
coeffs = sp.collect(expr.expand(), [a**2,u**2, a*u], evaluate=False)
new_expr_lst = []
for term, coeff in coeffs.items():
    print_math(sp.latex(term) + "=" + sp.latex(coeff))
    new_expr_lst.append((coeff.simplify())*term)
ua_expr = sum(new_expr_lst)
M_expr=sp.simplify((expr.subs({u:M*a}))*(gamma+1)*factor*rho).expand()
l_p_vl[i] =M_expr
M_expr

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

-M**2 + 2*M + gamma

In [580]:
B_collected

Matrix([
[ a**2 + a**2/(gamma - 1) - gamma*u**2/2,                      gamma*u,           -gamma],
[gamma**2*u**2/2 + gamma*(-a*u - u**2/2), -gamma**2*u - gamma*(-a - u), gamma**2 - gamma],
[ gamma**2*u**2/2 + gamma*(a*u - u**2/2),  -gamma**2*u - gamma*(a - u), gamma**2 - gamma]])

In [546]:
F_pp = sp.Matrix([0,0,0])
F_mm = sp.Matrix([0,0,0])

M = sp.symbols("M")

F_vl_p = (rho*a/4)*(M+1)**2 * sp.Matrix([
        1,
        ((gamma-1)*u+2*a)/gamma,
        ((gamma-1)*u+2*a)**2/(2*(gamma+1)*(gamma-1))
    ])

F_vl_m = (rho*a/4)*(M-1)**2 * sp.Matrix([
        1,
        ((gamma-1)*u-2*a)/gamma,
        ((gamma-1)*u-2*a)**2/(2*(gamma+1)*(gamma-1))
    ])

D = sp.diag( (gamma-1)*rho, a, -a)
D_inv = D.inv()
B = sp.simplify(D_inv*Qa_inv)*gamma
factor = 1/(rho*a**2)
B_collected = (B).applyfunc(lambda x: sp.collect(sp.apart(x, gamma), [u**2/2,u,]))
for i in range(B_collected.shape[0]): # iterate rows
    B_row = B_collected[i,:]
    F_factored = F_vl_p/F_vl_p[0]
    expr=(B_row[0]*F_vl_p[0] 
        + B_row[1]*F_vl_p[1] 
        + B_row[2]*F_vl_p[2]
        )
    coeffs = sp.collect(expr.expand(), [a**2,u**2, a*u], evaluate=False)
    new_expr_lst = []
    for term, coeff in coeffs.items():
        # print_math(sp.latex(term) + "=" + sp.latex(coeff))d
        new_expr_lst.append((coeff.simplify())*term)
    ua_expr = sum(new_expr_lst)
    M_expr=sp.simplify((expr.subs({u:M*a}))).expand()
    M_coeffs=sp.collect(M_expr, M, evaluate=False)
    M_expr_lst=[]
    for term, coeff in M_coeffs.items():
        
        M_expr_lst.append((sp.simplify(coeff))*term)
    M_expr = sum(M_expr_lst).simplify()
    print_math(sp.latex(M_expr))
    F_pp[i] =M_expr
for i in range(B_collected.shape[0]): # iterate rows
    B_row = B_collected[i,:]
    F_factored = F_vl_m/F_vl_m[0]
    expr=(B_row[0]*F_vl_m[0] 
        + B_row[1]*F_vl_m[1] 
        + B_row[2]*F_vl_m[2]
        )
    coeffs = sp.collect(expr.expand(), [a**2,u**2, a*u], evaluate=False)
    new_expr_lst = []
    for term, coeff in coeffs.items():
        # print_math(sp.latex(term) + "=" + sp.latex(coeff))d
        new_expr_lst.append((coeff.simplify())*term)
    ua_expr = sum(new_expr_lst)
    M_expr=sp.simplify((expr.subs({u:M*a}))).expand()
    M_coeffs=sp.collect(M_expr, M, evaluate=False)
    M_expr_lst=[]
    for term, coeff in M_coeffs.items():
        
        M_expr_lst.append((sp.simplify(coeff))*term)
    M_expr = sum(M_expr_lst).simplify()
    # print_math(sp.latex(M_expr))
    F_mm[i] =M_expr
F_pp
# F_mm

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Matrix([
[                          a*(-M**4 + M**2*(gamma + 3) + 2*M*(gamma + 1) + gamma)/(4*(gamma + 1))],
[a*(M**2*(M**2*(gamma - 1) - gamma + 3) + M*(5 - M**2)*(gamma + 1) + 4*gamma + 2)/(4*(gamma + 1))],
[          a*(M**2*(M**2*(gamma - 1) - gamma + 3) + M*(M**2 - 1)*(gamma + 1) - 2)/(4*(gamma + 1))]])

In [ ]:
x = (B*F_vl_p)
x_factored = (x[0]/F_vl_p[0]).subs({u:M*a}).simplify()
x_factored


(-M**2 + 2*M + gamma)/(rho*(gamma + 1))

In [577]:
((textbook[0]/F_vl_p[0]).simplify()-x_factored).simplify()

0

In [575]:
F_vl_p[0]

a*rho*(M + 1)**2/4

In [559]:
textbook = sp.Matrix([0,0,0])
textbook[0] = a/4*(M+1)**2 * ( 1 -(M-1)**2/(gamma+1))
textbook[1] = a/4*(M+1)**2 * ( 3 - M + (gamma-1)/(gamma+1)*(M-1)**2)
textbook[2] = a/2*(M+1)**2 *(M-1)/(gamma+1)*(1+(gamma-1)/2*M)
print((textbook[0]-(F_pp[0])).simplify())
print((textbook[1]-(F_pp[1])).simplify())
print((textbook[2]-(F_pp[2])).simplify())
textbook[0].expand().collect(M)

0
0
0


-M**4*a/(4*gamma + 4) + M**2*(a/4 + a/(2*gamma + 2)) + M*a/2 + a/4 - a/(4*gamma + 4)

In [536]:
textbook[0] = a/4*(M+1)**2 * ( 1 - (M-1)**2/(gamma+1))
F_vl_p_textbook = rho*a/4 * (M+1)**2
textbook_1 = (1/(rho*(gamma+1))*(-M**2 + 2*M + gamma)*F_vl_p_textbook)

textbook_1 -F_pp[0]


a*(M + 1)**2*(-M**2 + 2*M + gamma)/(4*(gamma + 1)) - (-M**2 + 2*M + gamma)/(rho*(gamma + 1))

In [538]:
(F_pp[0]*F_vl_p[0]-  textbook[0]).expand().simplify()

0

F_vl_p

In [ ]:
numerator,denominator = sp.fraction(textbook[0].simplify())
collect((numerator/a).expand(),M)
(numerator/F_vl_p[0]).expand().simplify()


4*(M**2 - 2*M + gamma + 2)/rho

In [ ]:
(textbook[0].simplify()-(F_pp[0]*F_vl_p[0]).simplify())

-a*(M + 1)**2*(-M**2 + 2*M + gamma)/(4*(gamma + 1)) + a*(M + 1)**2*(gamma + (M - 1)**2 + 1)/(4*(gamma + 1))

In [ ]:
F_vl_p[0]/(rho*(gamma+1))*(-M**2 + 2*M + gamma)

0

f3 = u*(E*rho + p); energy flux
unsuccessful derivation

In [ ]:
f3 = (F[2].subs({u: M*a}))
m3 = (f3 -M*a*p).simplify()
Mp, Mm = sp.symbols("M_p, M_m")
M_subs = {Mp:m1_p, Mm:m1_m}

m3 = (m3.subs(thermo_subs).subs({u: M*a})).simplify()
m3_p = (m3/M).subs({M**2: (a1*M**2 + b1*M + c1)})*Mp
m3_m = (m3/M).subs({M**2: (a2*M**2 + b2*M + c2)})*Mm
M3 = collect(((m3_p + m3_m).subs(M_subs)/(rho*a**3)).expand(),M)
coeffs = sp.collect(M3, M, evaluate=False)
coeff_eqns = []
for term,coeff in coeffs.items():
    # print(coeff)
    print(coeff)
    if term != M**3:
        coeff_eqns.append(sp.Eq(coeff.simplify(), 0))
    else:
        coeff_eqns.append(sp.Eq(coeff.simplify(), 1))
        
# sol3 = sp.solve(coeff_eqns, [a1, b1, c1, a2, b2,c2])
collect(((m3_p + m3_m)/(rho*a**3)).expand(),M)
m3

U2*U3*c1/(U1*a**3*rho) + U2*U3*c2/(U1*a**3*rho) + U2*c1*p/(U1*a**3*rho) + U2*c2*p/(U1*a**3*rho)
-p/(a**2*rho)
-c1*p/(a**2*rho) - c2*p/(a**2*rho) + U2*U3/(U1*a**3*rho) + U2*p/(U1*a**3*rho)


(-M*U1*a*p + U2*(U3 + p))/U1

In [ ]:
# cubic to split
m3 = M*(sp.Rational(1,2)*M**2 + 1/(gamma-1))

# split using your M+ and M-
m3_p = (m3/M).subs({M**2: (a1*M**2 + b1*M + c1)}) * Mp
m3_m = (m3/M).subs({M**2: (a2*M**2 + b2*M + c2)}) * Mm

# nondimensional combined expression
M3 = collect(((m3_p + m3_m).subs(M_subs)/(rho*a**3)).expand(), M)
coeffs = sp.collect(M3, M, evaluate=False)

coeff_eqns = []
for term, coeff in coeffs.items():
    if term == M**3:
        coeff_eqns.append(sp.Eq(coeff.simplify(), sp.Rational(1,2)))
    elif term == M**1:
        coeff_eqns.append(sp.Eq(coeff.simplify(), 1/(gamma-1)))
    else:
        coeff_eqns.append(sp.Eq(coeff.simplify(), 0))

# impose van Leer symmetry
# coeff_eqns.append(sp.Eq(a2, -a1))
coeff_eqns.append(sp.Eq(b2,  b1))
#coeff_eqns.append(sp.Eq(c2, -c1))

# solve
sol3 = sp.solve(coeff_eqns, [a1, b1, c1, a2, b2, c2], dict=True)
sol3


KeyboardInterrupt: 

In [ ]:
f3 = (F[2].subs({u: M*a}))
m3 = (f3 -M*a*p).simplify()

m3 = (m3.subs(thermo_subs).subs({u: M*a})).simplify()

p3 = f3 - m3
p3 = p3.subs(thermo_subs).subs({u: M*a}).simplify()
a1, b1, c1, d1, e1, a2, b2, c2, d2, e2 = sp.symbols("a1, b1, c1, d1, e1, a2, b2, c2, d2, e2")
Mp, Mm = sp.symbols("M_p, M_m")
M_subs = {Mp:m1_p, Mm:m1_m}
m3_p = m3.subs({M**2: Mp*(a1*M + b1)})
m3_m = m3.subs({M**2: Mm*(a2*M + b2)})

M3 = collect(((m3_p + m3_m)/(rho*a**3)).expand(),M)
# M3 = sp.collect(M3.subs(M_subs).expand(),M)
# m3_p =  Mp*(a1*M + b1*M)
# m3_m =  Mm*(a2*M + b2*M)

# M3 = collect(((m3_p + m3_m)).expand(),M)
M3 = sp.collect(M3.subs(M_subs).expand(),M)
collect(((m3_p + m3_m)*(2*(gamma-1)**2)/(rho*a**3)).expand(),M)
M3 = collect((M3.subs(M_subs)*(2*(gamma-1)**2)).simplify().expand(),M)
m3_coeffs = sp.collect(M3, M, evaluate=False)
coeff_eq = []
for term,coeff in m3_coeffs.items():
    print_math(sp.latex((m3_coeffs.get(term, 0).simplify())))
    if term != M**3:
        coeff_eq.append(sp.Eq(m3_coeffs.get(term, 0).simplify(), 0))
    else:
        coeff_eq.append(sp.Eq(m3_coeffs.get(term, 0).simplify(), 1))
sol3 = sp.solve(coeff_eq, [a1, b1, a2, b2])
p3_p = m1_p*(c1*M + d1)*p*a
p3_m = m1_m*(c2*M + d2)*p*a
P3 = sp.collect(((p3_p+p3_m)/p).expand(), M)
p3_coeffs = sp.collect(P3, M, evaluate=False)
for term,coeff in p3_coeffs.items():
    if term != 1:
        coeff_eq.append(sp.Eq(coeff, 0))
    else:
        coeff_eq.append(sp.Eq(coeff, 1))

print_math("\\dfrac{p}{p}=\\frac{p^+ + p^-}{p} = " + sp.latex(P3) + "= 1")
# 5. M**3
sol3 = sp.solve(coeff_eq, [a1, b1, c1, d1, a2, b2, c2, d2])
print_math("sol="+sp.latex(sol3))
m3_p = m3_p.subs(M_subs).subs(sol3)
m3_m = m3_m.subs(sol3)
p3_p = p3_p.subs(sol3)
p3_m = p3_m.subs(sol3)
f3_p = sp.simplify(((m3_p.subs(sol) + p3_p.subs(sol))))
f3_m = sp.simplify(((m3_m.subs(sol) + p3_m.subs(sol))))

f3_p_factored = sp.collect(((f3_p)/f1_p).subs({M:u/a}).subs({p:p_expanded}).simplify(), u)
f3_m_factored = sp.collect(((f3_m)/f1_m).subs({M:u/a}).subs({p:p_expanded}).simplify(), u)
print_math("f_3^+ = " + sp.latex(f3_p) +"=" + sp.latex(f1_p) +"\\cdot"+sp.latex(f3_p_factored))
# print_math("f_3^- = " + sp.latex(f2_m)+"=" + sp.latex(f1_p) +"\\cdot"+sp.latex(f3_m_factored))

numerator, denominator = m3_p.as_numer_denom()
numerator


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

{a1: (5*gamma - 4)/(gamma**3 - 2*gamma**2 + gamma),
 a2: (5*gamma - 4)/(gamma**3 - 2*gamma**2 + gamma),
 b1: -8/(gamma**2 - gamma),
 b2: 8/(gamma**2 - gamma)}

In [ ]:
# Symbols for coefficients
a1, b1, c1, d1, e1, a2, b2, c2, d2, e2 = sp.symbols("a1 b1 c1 d1 e1 a2 b2 c2 d2 e2")

# Original m3
f3 = F[2].subs(thermo_subs).subs({u: M*a})
m3 = f3 - (u*p).subs({u: M*a})

# Substitute M^2 as in F2
m3_p = m3.subs({M**2: m1_p*(a1*M + b1)})
m3_m = m3.subs({M**2: m1_m*(a2*M + b2)})

# Combine and normalize by rho*a^3
M_expr = collect(((m3_p + m3_m)/(a**3*rho)).expand(), M)
# print_math("\\frac{m^+ + m^-}{\\rho a^3}=" + sp.latex(M_expr))

# Extract coefficients for solving
coeffs = sp.collect(M_expr, M, evaluate=False)

# Set up equations (conservation + smoothness)
eq1 = sp.Eq(coeffs.get(M**3, 0).simplify(), 1)
eq2 = sp.Eq(coeffs.get(M**2, 0).simplify(), 0)
eq3 = sp.Eq(coeffs.get(M, 0).simplify(), 0)
eq4 = sp.Eq(coeffs.get(M**4, 0), 0)

# Solve for coefficients
sol3 = sp.solve([eq1, eq2, eq3, eq4], [a1, b1, a2, b2])
print_math("sol = " + sp.latex(sol3))

# Substitute solution back to get f3^+ and f3^-
f3_p_final = m3_p.subs(sol3).simplify()
f3_m_final = m3_m.subs(sol3).simplify()

# print_math("f_3^+ = " + sp.latex(f3_p_final))
# print_math("f_3^- = " + sp.latex(f3_m_final))

f3_p_final = sp.collect(((m3_p.subs(sol) )/f1_p).subs({M:u/a}).subs({p:p_expanded}).simplify(), u)

# print_math("f_3^+ = " +"\\cdot"+sp.latex(f3_p_factored))

sp.collect((m3_p.subs(sol3)/f1_p).simplify().subs({M:u/a}).subs({p:p_expanded}).simplify(), u)

<IPython.core.display.Math object>

u*(4*a**3 - (a + u)**2*(4*a + u*(-gamma**2 + gamma - 2)))/(gamma*(a + u)**2*(gamma - 1))

f3